In [1]:
CODE  = "/kaggle/input/datasets/theokorir/knee-mri-pipeline/knee_pipeline"
DATA  = "/kaggle/input/competitions/rsna-knee-abnormality-detection"
MODEL = "/kaggle/input/models/qwen-lm/qwen-3/transformers/8b/1"
W     = "/kaggle/working"
PREV  = "//kaggle/input/notebooks/theokorir/knee-mri-diagnosis-label"   # old output holding B.csv
!pip install -q langdetect bitsandbytes
!ls $CODE
!grep -c "calibrate_b" $CODE/labeler_combine.py   # must print >= 1, or an older version is attached

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 14.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 50.9 MB/s eta 0:00:00
cache_data.py	 infer.py	     labeler_rules.py  run_pipeline_example.sh
common.py	 labeler_combine.py  model.py	       tests
crop_preview.py  labeler_llm.py      README.md	       train.py
4


In [2]:
!python $CODE/labeler_rules.py --train_csv $DATA/train.csv --out $W/A.csv

saved /kaggle/working/A.csv (4407, 37)


In [3]:
!nvidia-smi

Sat Oct  3 14:44:35 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:
import pandas as pd, numpy as np
L = ["ACL","MCL","Medial Meniscus","Lateral Meniscus","Medial OA","Lateral OA","PF OA",
     "Effusion","Synovitis","Baker's","Contusion","Fracture"]
tr = pd.read_csv(f"{DATA}/train.csv")
gold = tr[tr[L].notna().all(axis=1)]
gold.to_csv(f"{W}/train_gold.csv", index=False); print(len(gold))   # expect 58

58


In [5]:
for v in ["v1", "v2"]:
    !python $CODE/labeler_llm.py --train_csv $W/train_gold.csv --out $W/B_gold_{v}.csv \
        --backend hf --model_path $MODEL --batch_size 4 --load_4bit --prompt {v}

`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|█| 399/399 [03:11<00:00,  2.08it/s, Materializing param=mo
cached: 0 | to label: 58
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
  labeled 32/58
  labeled 58/58
saved /kaggle/working/B_gold_v1.csv (58, 38) | parse failures: 0
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|█| 399/399 [00:10<00:00, 39.79it/s, Materializing param=mo
cached: 0 | to label: 58
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
  labeled 32/58
  labeled 58/58
saved /kaggle/working/B_gold_v2.csv (58, 38) | parse failures: 0


In [6]:
import os, shutil
USE_EXISTING = True          # True: reuse the v1 labels you already have
if USE_EXISTING:
    for f in ["B.csv", "B.csv.cache.jsonl"]:
        if os.path.exists(f"{PREV}/{f}"): shutil.copy(f"{PREV}/{f}", f"{W}/{f}")
    print(os.path.exists(f"{W}/B.csv"))
else:
    # about 7.9 h of GPU: run this notebook as Save & Run All, never interactively
    !python $CODE/labeler_llm.py --train_csv $DATA/train.csv --out $W/B.csv \
        --backend hf --model_path $MODEL --batch_size 4 --load_4bit --prompt v2

B = pd.read_csv(f"{W}/B.csv")
assert len(B) == 4407 and B.B_parse_ok.all(), "failed parses: re-run the B command; it retries them itself"

True


In [7]:
!python $CODE/labeler_combine.py combine --train_csv $DATA/train.csv --a $W/A.csv --b $W/B.csv \
    --out $W/labels_soft_v4.csv --reliability_out $W/reliability.csv --calibrate_b

calibrated P(gold | verdict), pooled: {'present': np.float64(0.73), 'absent': np.float64(0.02), 'uncertain': np.float64(0.33), 'not_mentioned': np.float64(0.12)}
mixing weights: A=0.20 B=0.34
Labeler A coverage: 1646 of 4407 reports are in languages A does not cover -> A confidence set to 0 there. Languages: {'tr': 546, 'hr': 406, 'el': 321, 'bg': 220, 'nl': 153}
                  n_gold  n_pos  n_neg  auc_A  auc_B  auc_mean  kappa_A_vs_B_all  prev_A  prev_B
label                                                                                           
ACL                   58     24     34  0.844  0.956     0.954             0.635   0.086   0.149
MCL                   58      9     49  0.811  0.929     0.943             0.477   0.049   0.129
Medial Meniscus       58     26     32  0.704  0.924     0.886             0.452   0.184   0.411
Lateral Meniscus      58     23     35  0.778  0.822     0.851             0.477   0.123   0.165
Medial OA             58     15     43  0.610  0.881